> **Proposal-stage prototype.** This notebook is the working prototype of the full pipeline (models, signals, portfolio, benchmarks, costs, bootstrap and indicator stability), run on Yahoo Finance and FRED data.
>
> Its logic is being ported into `src/`, step by step, as set out in `docs/bridge-plan.md`. The maintained pipeline is `notebooks/01_full_pipeline.ipynb`. This notebook is kept as evidence: later changes are added as switches in the configuration cell, never by deleting the original design. `INPUT_DESIGN = "original"` reproduces the proposal design (each model forecasts from its own selection); `"identical"` is the design agreed with the supervisor in October 2026 (LASSO and LightGBM only select; the linear model as a ridge fit, LightGBM and the LSTM all forecast from the union). Each run writes to `results/<date>_<commit>_<label>/` with a `run_info.json`.

# Neural Network-Driven Signal Generation for Systematic Portfolio Construction
### A Risk-Adjusted Extension of Optimized Technical Indicator Models

**MScFE Capstone Project (Research Track)**
Temitope Emmanuel Fagbuyi · Hart Bassey Ofigwe · Joseph Edet

This notebook implements the full signal-to-portfolio pipeline:

| Step | Section | Capstone objective |
|---|---|---|
| Download ETF prices and market data | 1 | 1. Data and feature engineering |
| Compute technical indicators and the 5-day label | 2 | 1. Data and feature engineering |
| Walk-forward splits with purging and embargo | 3 | 3. Out-of-sample validation |
| In-window feature selection; LASSO, GBM and LSTM models | 4 | 2. Indicator and signal modelling |
| Prediction metrics (accuracy, AUC, Brier, log-loss) | 5 | 3. Out-of-sample validation |
| Probabilities to volatility-scaled positions; benchmarks | 6 | 4. Portfolio construction and backtesting |
| Portfolio metrics, cost sensitivity, break-even cost | 7 | 5. Performance assessment |
| Block-bootstrap test on Sharpe differences | 8 | 5. Performance assessment |

**How to run:** in Google Colab choose *Runtime → Run all*. A full run (annual refits, 2008–2026) takes roughly 20–40 minutes on a standard CPU runtime, mostly for the LSTM. Set `QUICK_MODE = True` in the configuration cell for a faster first pass (refits every 3 years, fewer LSTM epochs).

**Status:** working outline. Sections marked **TODO** are planned for the next project stage.

In [ ]:
# Colab already includes pandas, numpy, scikit-learn, matplotlib and TensorFlow.
!pip -q install yfinance lightgbm

## 0. Configuration
All design choices live here so they can be changed and reported in one place.

In [ ]:
import os, sys, json, time, random, subprocess, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, accuracy_score

warnings.filterwarnings("ignore")

# ---------- Universe and sample ----------
TICKERS   = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "QQQ", "IWM"]  # tradable
BENCHMARK = "SPY"                          # benchmark only
START, END = "2000-01-01", "2026-07-01"    # yfinance END is exclusive -> data to 30 Jun 2026
FRED_SERIES = {"DTB3": "tbill_3m", "VIXCLS": "vix", "T10Y3M": "term_spread"}

# ---------- Target and validation ----------
HORIZON         = 5    # predict sign of the 5-trading-day return
EXEC_LAG        = 1    # signal at close t, trade at close t+1; label runs close t+1 -> close t+1+HORIZON
FIRST_TEST_YEAR = 2008
VAL_MONTHS      = 12   # last 12 months of each training window used for tuning / early stopping
EMBARGO         = 5    # extra trading days left empty after the purge
PURGE_GAP       = EXEC_LAG + HORIZON + EMBARGO   # trading days between training and validation/test data

# ---------- Models ----------
LASSO_CS      = [0.002, 0.005, 0.01, 0.05, 0.1]  # inverse L1 penalty strengths for the SELECTION fit
TOP_K_GBM     = 15      # GBM keeps its top-15 features by importance
# "identical": LASSO and LightGBM only select; ridge, LightGBM and LSTM all forecast from the union
# (supervisor, Oct 2026). "original": each model forecasts from its own selection (proposal design).
INPUT_DESIGN  = "identical"
LSTM_LOOKBACK = 20      # LSTM sees the last 20 trading days of features
RUN_LSTM      = True
QUICK_MODE    = False   # True: refit every 3 years and train the LSTM for fewer epochs
REFIT_EVERY   = 3 if QUICK_MODE else 1
LSTM_EPOCHS   = 8 if QUICK_MODE else 25

# ---------- Portfolio ----------
VOL_TARGET     = 0.10   # 10% annualised portfolio volatility target
VOL_LOOKBACK   = 63     # ~3 months for volatility and covariance estimates
MAX_WEIGHT     = 0.25   # max 25% per ETF
NO_TRADE_BAND  = 0.02   # skip weight changes below 2 percentage points
P_HIGH         = 0.90   # full position at the 90th percentile of validation-period forecasts
MIN_EDGE       = 0.01   # floor on (p90 - base rate); below it the model stays flat for that window
COST_BPS       = 5      # base-case cost, basis points per side
COST_GRID      = [0, 2, 5, 10]

# ---------- Statistics ----------
N_BOOT    = 10_000
BOOT_BLOCK = 20        # mean block length (trading days) for the stationary bootstrap

# ---------- Run settings ----------
RUN_LABEL   = "lstm-multiseed"
DATA_SOURCE = "cache"  # "cache": data/raw/*.parquet from src/data.py; "download": Yahoo + FRED; "synthetic": smoke test
# monotonic() stops while the machine sleeps, so run times are not inflated by sleep.
T_START, CPU_START = time.monotonic(), time.process_time()

def _git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return ""

# Shared settings come from config.yaml when the notebook runs inside the repo, so the
# ridge grid, placebo settings and seed are defined in one place.
REPO = Path(_git("rev-parse", "--show-toplevel") or ".")
CFG = {}
if (REPO / "config.yaml").exists():
    import yaml
    CFG = yaml.safe_load(open(REPO / "config.yaml"))
SEED     = CFG.get("seed", 42)
RIDGE_CS = CFG.get("models", {}).get("lasso", {}).get(
    "ridge_C_grid", [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0])   # ridge FORECASTING fit
PLACEBO  = CFG.get("stats", {}).get("placebo", {"method": "circular_shift", "n_draws": 1000,
                                                "quick_draws": 100, "min_shift_weeks": 52})
N_PLACEBO = PLACEBO["quick_draws"] if QUICK_MODE else PLACEBO["n_draws"]

# Table labels. Code keys stay lasso/gbm/lstm; under the identical design the "lasso" key is the
# ridge forecasting fit on the union, so it is labelled as such.
LABEL = {"lasso": "Linear (L1 selection, ridge on union)" if INPUT_DESIGN == "identical" else "LASSO",
         "gbm": "LightGBM", "lstm": "LSTM"}

# One LSTM per seed: a single training run can be unrepresentative. The first seed is the main
# "lstm"; the others are extra models "lstm_s<seed>" that go through the same signals, backtest,
# bootstrap and placebo. Quick mode keeps two seeds.
LSTM_SEEDS = CFG.get("models", {}).get("lstm", {}).get("seeds", [SEED])
if QUICK_MODE:
    LSTM_SEEDS = LSTM_SEEDS[:2]
LSTM_KEYS = {("lstm" if i == 0 else f"lstm_s{sd}"): sd for i, sd in enumerate(LSTM_SEEDS)}
LABEL.update({k: f"LSTM (seed {sd})" for k, sd in LSTM_KEYS.items() if k != "lstm"})

np.random.seed(SEED); random.seed(SEED)

# Results folder named after the code that produced it: results/<date>_<commit>_<label>/
COMMIT = _git("rev-parse", "--short", "HEAD") or "nogit"
if _git("status", "--porcelain", "--untracked-files=no"):
    print("WARNING: uncommitted changes; the folder name will not identify the code exactly.")
OUT = REPO / "results" / f"{pd.Timestamp.now():%Y-%m-%d}_{COMMIT}_{RUN_LABEL}"
FIG, LARGE = OUT / "figures", OUT / "large"
for d in (OUT, FIG, LARGE):
    d.mkdir(parents=True, exist_ok=True)

print(f"Purge + embargo gap: {PURGE_GAP} trading days | refit every {REFIT_EVERY} year(s)")
print(f"Input design: {INPUT_DESIGN} | data: {DATA_SOURCE} | ridge grid: {RIDGE_CS} | LSTM seeds: {LSTM_SEEDS}")
print(f"Output: {OUT}")

## 1. Data
* **Prices and volume:** Yahoo Finance via `yfinance`, split- and dividend-adjusted (`auto_adjust=True`).
* **Risk-free rate, VIX, term spread:** FRED (`DTB3`, `VIXCLS`, `T10Y3M`), no API key needed.
* FRED values are **lagged one trading day** so a feature on date *t* only uses information published by *t*.

In [ ]:
def download_prices(tickers, start, end):
    # Returns a dict of DataFrames: Open, High, Low, Close, Volume (dates x tickers).
    for attempt in range(3):   # Yahoo occasionally rate-limits; retry before failing
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True,
                          progress=False, group_by="column", threads=True)
        if not raw.empty and raw["Close"].notna().any().all():
            break
        time.sleep(10)
    return {f: raw[f][tickers].copy() for f in ["Open", "High", "Low", "Close", "Volume"]}

def download_fred(series_ids):
    out = {}
    for sid, name in series_ids.items():
        df = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}")
        df.columns = ["date", name]
        df["date"] = pd.to_datetime(df["date"])
        out[name] = pd.to_numeric(df[name], errors="coerce").set_axis(df["date"])
    return pd.DataFrame(out)

In [ ]:
ALL = TICKERS + [BENCHMARK]
FRED_NAMES = {"DTB3": "tbill_3m", "VIX": "vix", "TERM_SPREAD": "term_spread"}

def from_repo_layout(prices, macro_raw):
    # src/data.py stores raw OHLC plus the adjusted close. Scaling open/high/low by adj/close makes
    # every price dividend-adjusted, as yfinance's auto_adjust=True does for the download path.
    adj = prices["adj_close"] / prices["close"]
    px = {"Open": prices["open"] * adj, "High": prices["high"] * adj, "Low": prices["low"] * adj,
          "Close": prices["adj_close"], "Volume": prices["volume"]}
    return {k: v[ALL].copy() for k, v in px.items()}, macro_raw.rename(columns=FRED_NAMES)

if DATA_SOURCE == "cache":
    px, fred = from_repo_layout(pd.read_parquet(REPO / "data/raw/prices.parquet"),
                                pd.read_parquet(REPO / "data/raw/macro.parquet"))
elif DATA_SOURCE == "synthetic":
    # Smoke test only: src.data.make_synthetic returns macro already lagged once.
    sys.path.insert(0, str(REPO))
    from src.data import make_synthetic
    px, fred = from_repo_layout(*make_synthetic(CFG, n_days=6700))
else:
    px = download_prices(ALL, START, END)
    fred = download_fred(FRED_SERIES)

dates = px["Close"].dropna(how="all").index
for f in px: px[f] = px[f].reindex(dates)
macro = fred.reindex(dates.union(fred.index)).sort_index().ffill().reindex(dates).shift(1)  # 1-day lag
rf_daily = (macro["tbill_3m"] / 100 / 252).fillna(0)

summary = pd.DataFrame({
    "first_date": px["Close"].apply(lambda s: s.first_valid_index()),
    "last_date":  px["Close"].apply(lambda s: s.last_valid_index()),
    "n_days":     px["Close"].count(),
    "missing_after_start": px["Close"].apply(lambda s: s.loc[s.first_valid_index():].isna().sum()),
})
print(f"Trading days: {len(dates)} ({dates[0].date()} to {dates[-1].date()})")
summary

## 2. Feature engineering
All indicators are expressed as ratios, returns or bounded oscillators so they are comparable across ETFs. Standardisation (z-scoring) is fitted **inside each training window** in Section 4, never on the full sample.

| Family | Indicators (parameters) |
|---|---|
| Trend | Price / SMA − 1 (10, 20, 50, 200); MACD (12, 26, 9) line, signal, histogram (÷ price); ADX (14) |
| Momentum | Returns over 5, 21, 63, 126, 252 days; RSI (14); Stochastic %K (14), %D (3); Williams %R (14); ROC (10) |
| Volatility | Realised volatility (21, 63); ATR (14) ÷ price; Bollinger %B and bandwidth (20, 2σ) |
| Volume | 20-day OBV change ÷ 20-day volume; log-volume z-score (20); Chaikin Money Flow (20); MFI (14) |
| Market context | VIX level and 5-day change; SPY 21-day return; 10y–3m term spread |

**Label:** `y = 1` if the ETF's return from the close on *t+1* (execution) to the close on *t+1+5* is positive. This aligns the label with the return the portfolio actually earns.

In [ ]:
def wilder(x, n):
    return x.ewm(alpha=1 / n, adjust=False).mean()

def true_range(h, l, c):
    return pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)

def rsi(c, n=14):
    d = c.diff()
    return 100 - 100 / (1 + wilder(d.clip(lower=0), n) / wilder(-d.clip(upper=0), n))

def adx(h, l, c, n=14):
    up, dn = h.diff(), -l.diff()
    plus_dm  = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=c.index)
    minus_dm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=c.index)
    atr_ = wilder(true_range(h, l, c), n)
    pdi, mdi = 100 * wilder(plus_dm, n) / atr_, 100 * wilder(minus_dm, n) / atr_
    return wilder(100 * (pdi - mdi).abs() / (pdi + mdi), n)

def ticker_features(o, h, l, c, v):
    f = pd.DataFrame(index=c.index)
    logret = np.log(c).diff()
    # Trend
    for n in [10, 20, 50, 200]:
        f[f"px_sma{n}"] = c / c.rolling(n).mean() - 1
    macd = (c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()) / c
    f["macd"] = macd
    f["macd_signal"] = macd.ewm(span=9, adjust=False).mean()
    f["macd_hist"] = macd - f["macd_signal"]
    f["adx14"] = adx(h, l, c, 14)
    # Momentum
    for n in [5, 21, 63, 126, 252]:
        f[f"ret{n}"] = c.pct_change(n)
    f["rsi14"] = rsi(c, 14)
    lo14, hi14 = l.rolling(14).min(), h.rolling(14).max()
    f["stoch_k14"] = 100 * (c - lo14) / (hi14 - lo14)
    f["stoch_d3"] = f["stoch_k14"].rolling(3).mean()
    f["willr14"] = -100 * (hi14 - c) / (hi14 - lo14)
    f["roc10"] = c.pct_change(10)
    # Volatility
    f["rv21"] = logret.rolling(21).std() * np.sqrt(252)
    f["rv63"] = logret.rolling(63).std() * np.sqrt(252)
    f["atr14"] = wilder(true_range(h, l, c), 14) / c
    ma20, sd20 = c.rolling(20).mean(), c.rolling(20).std()
    f["bb_pctb"] = (c - (ma20 - 2 * sd20)) / (4 * sd20)
    f["bb_width"] = 4 * sd20 / ma20
    # Volume
    obv = (np.sign(c.diff()).fillna(0) * v).cumsum()
    f["obv_chg20"] = (obv - obv.shift(20)) / v.rolling(20).sum()
    lv = np.log(v.replace(0, np.nan))
    f["vol_z20"] = (lv - lv.rolling(20).mean()) / lv.rolling(20).std()
    mfm = ((c - l) - (h - c)) / (h - l).replace(0, np.nan)
    f["cmf20"] = (mfm * v).rolling(20).sum() / v.rolling(20).sum()
    tp = (h + l + c) / 3
    mf = tp * v
    pos, neg = mf.where(tp > tp.shift(), 0.0), mf.where(tp < tp.shift(), 0.0)
    f["mfi14"] = 100 - 100 / (1 + pos.rolling(14).sum() / neg.rolling(14).sum())
    return f

def forward_label(c):
    entry, exit_ = c.shift(-EXEC_LAG), c.shift(-(EXEC_LAG + HORIZON))
    fwd = exit_ / entry - 1
    return fwd.gt(0).astype(float).where(fwd.notna())

In [ ]:
spy = px["Close"][BENCHMARK]
market = pd.DataFrame({
    "vix": macro["vix"],
    "vix_chg5": macro["vix"] - macro["vix"].shift(5),
    "spy_ret21": spy.pct_change(21),
    "term_spread": macro["term_spread"],
}, index=dates)

frames = []
for tk in TICKERS:
    f = ticker_features(*(px[k][tk] for k in ["Open", "High", "Low", "Close", "Volume"]))
    f = f.join(market)
    f["y"] = forward_label(px["Close"][tk])
    f["ticker"] = tk
    frames.append(f)

panel = (pd.concat(frames).rename_axis("date").reset_index()
         .set_index(["date", "ticker"]).sort_index().replace([np.inf, -np.inf], np.nan))
FEATURES = [c for c in panel.columns if c != "y"]
panel = panel.dropna(subset=FEATURES)          # removes warm-up rows (e.g. IWM until ~May 2001)

print(f"{len(FEATURES)} features | {len(panel):,} ETF-day rows")
print("First usable date per ETF:")
print(panel.reset_index().groupby("ticker")["date"].min().dt.date.to_string())
print(f"Share of positive 5-day labels: {panel['y'].mean():.3f}")

## 3. Walk-forward splits with purging and embargo
* **Expanding window**, refitted every `REFIT_EVERY` year(s); the first test year is 2008.
* Inside each training window, the **last 12 months** are the validation set (hyperparameter choice and early stopping).
* Because 5-day labels overlap, a gap of `PURGE_GAP` = 1 (execution lag) + 5 (horizon) + 5 (embargo) = **11 trading days** separates training from validation and validation from test. No training label can use a price from the validation or test period.

In [ ]:
def walk_forward_splits(all_dates, first_test_year, refit_every, val_months, gap):
    all_dates = pd.DatetimeIndex(sorted(all_dates))
    last_year = all_dates[-1].year
    splits = []
    for y0 in range(first_test_year, last_year + 1, refit_every):
        test = all_dates[(all_dates.year >= y0) & (all_dates.year < y0 + refit_every)]
        if len(test) == 0:
            continue
        p0 = all_dates.get_loc(test[0])
        train_all = all_dates[: max(p0 - gap, 0)]
        val_start = train_all[-1] - pd.DateOffset(months=val_months)
        val = train_all[train_all > val_start]
        pv = all_dates.get_loc(val[0])
        train = all_dates[: max(pv - gap, 0)]
        splits.append({"label": f"{y0}" if refit_every == 1 else f"{y0}-{test[-1].year}",
                       "train": train, "val": val, "test": test})
    return splits

panel_dates = panel.index.get_level_values("date").unique()
splits = walk_forward_splits(panel_dates, FIRST_TEST_YEAR, REFIT_EVERY, VAL_MONTHS, PURGE_GAP)
pd.DataFrame([{"test": s["label"],
               "train": f"{s['train'][0].date()} to {s['train'][-1].date()}",
               "validation": f"{s['val'][0].date()} to {s['val'][-1].date()}",
               "test_period": f"{s['test'][0].date()} to {s['test'][-1].date()}"} for s in splits])

## 4. Feature selection and models (inside each window)
Everything below is fitted **only on that window's training data**:

1. `StandardScaler` fitted on training rows.
2. **LASSO** – L1-penalised logistic regression; penalty chosen by validation log-loss. Its non-zero coefficients are its selected indicators.
3. **Gradient boosting (LightGBM)** – fitted on all indicators, ranked by gain importance, then refitted on the **top 15**, with early stopping on validation.
4. **LSTM** – uses the **union** of the LASSO and GBM selections, reading sequences of the last 20 days; early stopping on validation.

**Input design switch.** With `INPUT_DESIGN = "identical"` (supervisor, October 2026), steps 2 and 3 only **select**. All three forecasting models are then fitted on the union: the linear model as an **L2 (ridge)** logistic regression that keeps every union feature, with its penalty chosen on validation log-loss over `RIDGE_CS` from `config.yaml`; LightGBM with the same settings and no further selection; and the LSTM as above. `"original"` keeps the proposal design above. The chosen penalties are logged per window, with a count of how often each sits on the edge of its grid.

Selected indicators are recorded per window to measure selection stability (Section 5).

Each window also records the two inputs to the position mapping (Section 6), both from data **before** the test period: the training-window **base rate** `b` (share of positive 5-day labels) and `p90`, the 90th percentile of the model's validation-period forecasts.

In [ ]:
def fit_lasso(Xtr, ytr, Xva, yva, features):
    best = None
    for C in LASSO_CS:
        m = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000)
        m.fit(Xtr, ytr)
        coef = m.coef_.ravel()
        if not np.any(coef != 0):
            continue
        ll = log_loss(yva, m.predict_proba(Xva)[:, 1])
        if best is None or ll < best["ll"]:
            best = {"ll": ll, "C": C, "model": m, "selected": [f for f, b in zip(features, coef) if b != 0]}
    if best is None:   # every penalty removed all features: fall back to the weakest penalty
        m = LogisticRegression(penalty="l1", solver="liblinear", C=max(LASSO_CS) * 10, max_iter=5000).fit(Xtr, ytr)
        best = {"ll": np.nan, "C": max(LASSO_CS) * 10, "model": m,
                "selected": [f for f, b in zip(features, m.coef_.ravel()) if b != 0] or list(features)}
    return best

GBM_PARAMS = dict(n_estimators=2000, learning_rate=0.02, num_leaves=15, min_child_samples=200,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                  random_state=SEED, verbose=-1)

def fit_gbm(Xtr, ytr, Xva, yva, features):
    stop = [lgb.early_stopping(100, verbose=False)]
    full = lgb.LGBMClassifier(**GBM_PARAMS).fit(Xtr, ytr, eval_set=[(Xva, yva)],
                                                eval_metric="binary_logloss", callbacks=stop)
    imp = pd.Series(full.booster_.feature_importance("gain"), index=features).sort_values(ascending=False)
    top = list(imp.index[:TOP_K_GBM])
    idx = [features.index(f) for f in top]
    model = lgb.LGBMClassifier(**GBM_PARAMS).fit(Xtr[:, idx], ytr, eval_set=[(Xva[:, idx], yva)],
                                                 eval_metric="binary_logloss", callbacks=stop)
    return {"model": model, "selected": top, "idx": idx}

def fit_ridge_forecast(Xtr, ytr, Xva, yva):
    # Identical design: the linear forecasting fit on the union is L2, so it keeps every union feature.
    best = None
    for C in RIDGE_CS:
        m = LogisticRegression(penalty="l2", solver="lbfgs", C=C, max_iter=5000).fit(Xtr, ytr)
        ll = log_loss(yva, m.predict_proba(Xva)[:, 1])
        if best is None or ll < best["ll"]:
            best = {"ll": ll, "C": C, "model": m}
    return best

def fit_gbm_forecast(Xtr, ytr, Xva, yva):
    # Identical design: LightGBM on the union, same settings, no further selection.
    stop = [lgb.early_stopping(100, verbose=False)]
    return lgb.LGBMClassifier(**GBM_PARAMS).fit(Xtr, ytr, eval_set=[(Xva, yva)],
                                                eval_metric="binary_logloss", callbacks=stop)


In [ ]:
if RUN_LSTM:
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")
    from tensorflow import keras
    tf.random.set_seed(SEED)

def make_sequences(scaled, keep_dates, L):
    # scaled: DataFrame (date, ticker) x features, already standardised.
    # Returns windows of L days ending on each date in keep_dates, and their index.
    keep_dates = pd.DatetimeIndex(keep_dates)
    X_parts, idx_parts = [], []
    for tk, g in scaled.groupby(level="ticker", sort=False):
        arr = g.to_numpy(dtype="float32")
        d = g.index.get_level_values("date")
        pos = np.where(d.isin(keep_dates))[0]
        pos = pos[pos >= L - 1]
        if len(pos) == 0:
            continue
        win = np.lib.stride_tricks.sliding_window_view(arr, (L, arr.shape[1]))[:, 0]
        X_parts.append(win[pos - L + 1])
        idx_parts.append(g.index[pos])
    return np.concatenate(X_parts), idx_parts[0].append(idx_parts[1:])

def build_lstm(n_features):
    model = keras.Sequential([
        keras.layers.Input(shape=(LSTM_LOOKBACK, n_features)),
        keras.layers.LSTM(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    return model

def fit_predict_lstm(scaled, y, split, feats, seed=SEED):
    sub = scaled[feats]
    Xtr, itr = make_sequences(sub, split["train"], LSTM_LOOKBACK)
    Xva, iva = make_sequences(sub, split["val"], LSTM_LOOKBACK)
    Xte, ite = make_sequences(sub, split["test"], LSTM_LOOKBACK)
    ytr, yva = y.reindex(itr).to_numpy(), y.reindex(iva).to_numpy()
    mtr, mva = ~np.isnan(ytr), ~np.isnan(yva)
    # tf.random.set_seed alone does not fix Keras 3's weight initialisation, which draws from Python's
    # global RNG, so results depended on how many fits came before. Seed everything Keras uses, then
    # restore the global RNG state so the LASSO (liblinear) and LightGBM fits are unaffected.
    py_state, np_state = random.getstate(), np.random.get_state()
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    model = build_lstm(len(feats))
    model.fit(Xtr[mtr], ytr[mtr], validation_data=(Xva[mva], yva[mva]), epochs=LSTM_EPOCHS,
              batch_size=512, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    p_val = model.predict(Xva, batch_size=2048, verbose=0).ravel()
    p_test = pd.Series(model.predict(Xte, batch_size=2048, verbose=0).ravel(), index=ite)
    random.setstate(py_state); np.random.set_state(np_state)
    return p_val, p_test

In [ ]:
preds, selections, edges, tuning_rows = [], [], [], []
t_start = time.monotonic()

for s in splits:
    t0 = time.monotonic()
    d = panel.index.get_level_values("date")
    tr = panel[d.isin(s["train"]) & panel["y"].notna()]
    va = panel[d.isin(s["val"]) & panel["y"].notna()]
    te = panel[d.isin(s["test"])]

    scaler = StandardScaler().fit(tr[FEATURES])
    Xtr, Xva, Xte = (scaler.transform(x[FEATURES]) for x in (tr, va, te))
    ytr, yva = tr["y"].to_numpy(), va["y"].to_numpy()

    lasso = fit_lasso(Xtr, ytr, Xva, yva, FEATURES)
    gbm = fit_gbm(Xtr, ytr, Xva, yva, FEATURES)
    union = [f for f in FEATURES if f in set(lasso["selected"]) | set(gbm["selected"])]
    uidx = [FEATURES.index(f) for f in union]

    if INPUT_DESIGN == "identical":
        # The selection fits above only choose features; every forecasting model sees the union.
        ridge = fit_ridge_forecast(Xtr[:, uidx], ytr, Xva[:, uidx], yva)
        lin_model, lin_cols = ridge["model"], uidx
        gbm_model, gbm_cols = fit_gbm_forecast(Xtr[:, uidx], ytr, Xva[:, uidx], yva), uidx
    else:
        ridge = None
        lin_model, lin_cols = lasso["model"], list(range(len(FEATURES)))
        gbm_model, gbm_cols = gbm["model"], gbm["idx"]

    out = pd.DataFrame(index=te.index)
    out["y"] = te["y"]
    out["p_lasso"] = lin_model.predict_proba(Xte[:, lin_cols])[:, 1]
    out["p_gbm"] = gbm_model.predict_proba(Xte[:, gbm_cols])[:, 1]
    val_preds = {"lasso": lin_model.predict_proba(Xva[:, lin_cols])[:, 1],
                 "gbm": gbm_model.predict_proba(Xva[:, gbm_cols])[:, 1]}
    tuning_rows.append({"window": s["label"], "lasso_C": lasso["C"],
                        "ridge_C": ridge["C"] if ridge else np.nan,
                        "n_lasso": len(lasso["selected"]), "n_union": len(union)})
    if RUN_LSTM:
        # Scale every row with this window's scaler so LSTM look-back windows are consistent.
        scaled_all = pd.DataFrame(scaler.transform(panel[FEATURES]), index=panel.index, columns=FEATURES)
        for key, sd in LSTM_KEYS.items():
            val_preds[key], p_test = fit_predict_lstm(scaled_all, panel["y"], s, union, seed=sd)
            out[f"p_{key}"] = p_test.reindex(out.index)

    # Position-mapping inputs, all from data before the test period:
    #   b   = training-window base rate;  p90 = 90th percentile of validation-period forecasts.
    b = ytr.mean()
    for m, pv in val_preds.items():
        p90 = np.quantile(pv, P_HIGH)
        p = out[f"p_{m}"]
        flat = (p90 - b) < MIN_EDGE                       # edge too small: stay flat all window
        out[f"s_{m}"] = 0.0 if flat else ((p - b) / (p90 - b)).clip(0, 1)   # main signal
        out[f"sb_{m}"] = ((p - b) / (1 - b)).clip(0, 1)                      # base-rate centred only
        out[f"sr_{m}"] = (2 * p - 1).clip(0, 1)                              # original 2p - 1
        edges.append({"window": s["label"], "model": LABEL[m], "base_rate": b,
                      "p90": p90, "edge": p90 - b, "flat": flat})

    preds.append(out)
    for name, feats in [("LASSO", lasso["selected"]), ("GBM", gbm["selected"]), ("LSTM (union)", union)]:
        selections.append({"window": s["label"], "model": name, "n": len(feats), "features": feats})
    print(f"{s['label']}: train {len(tr):,} | val {len(va):,} | test {len(te):,} | "
          f"LASSO select C={lasso['C']} ({len(lasso['selected'])} feats) | GBM top {len(gbm['selected'])} | "
          f"union {len(union)} feats" + (f" | ridge C={ridge['C']}" if ridge else "") + f" | {time.monotonic() - t0:.0f}s")

pred = pd.concat(preds).sort_index()
edge_table = pd.DataFrame(edges)
edge_table.to_csv(OUT / "signal_edges.csv", index=False)
pred.to_csv(LARGE / "predictions.csv")
MODELS = [m for m in ["lasso", "gbm", *LSTM_KEYS] if f"p_{m}" in pred]
print(f"Walk-forward finished in {(time.monotonic() - t_start) / 60:.1f} min")

In [ ]:
# Tuning hygiene: how often does each tuned penalty land on the edge of its grid?
# A value on an edge means the best penalty may lie outside the grid.
tuning = pd.DataFrame(tuning_rows).set_index("window")
edge_rows = []
for col, grid in {"lasso_C": LASSO_CS, "ridge_C": RIDGE_CS}.items():
    chosen = tuning[col].dropna()
    if chosen.empty:
        continue
    edge_rows.append({"parameter": col, "grid": str(grid), "windows": len(chosen),
                      "at_lower_edge": int((chosen <= min(grid)).sum()),
                      "at_upper_edge": int((chosen >= max(grid)).sum()),   # includes the x10 fallback
                      "chosen": "; ".join(f"{c:g}: {n}" for c, n in chosen.value_counts().sort_index().items())})
tuning_edges = pd.DataFrame(edge_rows).set_index("parameter")
tuning.to_csv(OUT / "tuning_by_window.csv")
tuning_edges.to_csv(OUT / "tuning_edges.csv")
for p, r in tuning_edges.iterrows():
    if r["at_lower_edge"] + r["at_upper_edge"] > 3:
        print(f"WARNING: {p} sits on a grid edge in {r['at_lower_edge'] + r['at_upper_edge']} of {r['windows']} windows")
tuning_edges


## 5. Out-of-sample prediction quality
Accuracy uses a 0.5 cut-off and should be read against the **base rate** (share of positive labels), since equity returns are positive more often than not. AUC measures ranking ability; Brier score and log-loss measure probability quality (calibration).

In [ ]:
def prediction_metrics(df, models):
    rows = []
    lab = df.dropna(subset=["y"])
    for m in models:
        p = lab[f"p_{m}"].clip(1e-6, 1 - 1e-6)
        rows.append({"model": LABEL[m], "n": len(lab), "base_rate": lab["y"].mean(),
                     "accuracy": accuracy_score(lab["y"], p > 0.5),
                     "AUC": roc_auc_score(lab["y"], p),
                     "Brier": brier_score_loss(lab["y"], p),
                     "Brier (base rate)": brier_score_loss(lab["y"], np.full(len(lab), lab["y"].mean())),
                     "log_loss": log_loss(lab["y"], p),
                     "mean_p": p.mean(), "sd_p": p.std()})
    return pd.DataFrame(rows).set_index("model").round(4)

pred_table = prediction_metrics(pred, MODELS)
pred_table.to_csv(OUT / "prediction_metrics.csv")
pred_table

In [ ]:
# AUC by test year and by ETF: is predictability stable over time and across sectors?
lab = pred.dropna(subset=["y"]).reset_index()
lab["year"] = lab["date"].dt.year
auc_year = lab.groupby("year").apply(lambda g: pd.Series({LABEL[m]: roc_auc_score(g["y"], g[f"p_{m}"]) for m in MODELS}))
auc_etf = lab.groupby("ticker").apply(lambda g: pd.Series({LABEL[m]: roc_auc_score(g["y"], g[f"p_{m}"]) for m in MODELS}))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
auc_year.plot(ax=ax[0], marker="o"); ax[0].axhline(0.5, color="grey", ls="--"); ax[0].set_title("Out-of-sample AUC by year")
auc_etf.plot.bar(ax=ax[1]); ax[1].axhline(0.5, color="grey", ls="--"); ax[1].set_title("Out-of-sample AUC by ETF")
ax[1].set_ylim(0.4, 0.6); plt.tight_layout(); fig.savefig(FIG / "auc_by_year_etf.png", dpi=120); plt.show()
auc_year.round(4).to_csv(OUT / "auc_by_year.csv")

In [ ]:
# Selection stability: (1) how often each indicator is selected, (2) average overlap between
# consecutive windows (Jaccard, 0 = completely different, 1 = identical), (3) indicator families.
FAMILY = {**{f: "Trend" for f in ["px_sma10", "px_sma20", "px_sma50", "px_sma200", "macd", "macd_signal", "macd_hist", "adx14"]},
          **{f: "Momentum" for f in ["ret5", "ret21", "ret63", "ret126", "ret252", "rsi14", "stoch_k14", "stoch_d3", "willr14", "roc10"]},
          **{f: "Volatility" for f in ["rv21", "rv63", "atr14", "bb_pctb", "bb_width"]},
          **{f: "Volume" for f in ["obv_chg20", "vol_z20", "cmf20", "mfi14"]},
          **{f: "Market context" for f in ["vix", "vix_chg5", "spy_ret21", "term_spread"]}}
assert set(FAMILY) == set(FEATURES), set(FEATURES) ^ set(FAMILY)

sel = pd.DataFrame(selections)
n_win = sel["window"].nunique()

# (1) Selection frequency per indicator
stability = pd.DataFrame({
    m: pd.Series(Counter(f for feats in sel.loc[sel["model"] == m, "features"] for f in feats)) / n_win
    for m in ["LASSO", "GBM"]}).reindex(FEATURES).fillna(0).sort_values("GBM", ascending=False)
stability.insert(0, "family", stability.index.map(FAMILY))
stability.to_csv(OUT / "feature_selection_stability.csv")

# (2) Jaccard overlap between consecutive windows
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else np.nan

# (3) Share of each model's selected indicators that falls in each family, averaged over windows
def family_shares(feats):
    return pd.Series([FAMILY[f] for f in feats]).value_counts(normalize=True)

summary_rows, fam_rows = [], {}
for m in ["LASSO", "GBM"]:
    lists = sel.loc[sel["model"] == m, "features"].tolist()
    overlaps = [jaccard(a, b) for a, b in zip(lists[:-1], lists[1:])]
    fam = pd.DataFrame([family_shares(f) for f in lists]).fillna(0).mean()
    fam_rows[m] = fam
    top = stability[m].idxmax()
    summary_rows.append({"model": m, "avg. indicators selected": np.mean([len(f) for f in lists]),
                         "avg. overlap (Jaccard)": np.nanmean(overlaps) if overlaps else np.nan,
                         "most selected family": fam.idxmax(),
                         "most selected indicator": f"{top} ({stability.loc[top, m]:.0%} of windows)"})
stab_summary = pd.DataFrame(summary_rows).set_index("model")
family_table = pd.DataFrame(fam_rows).reindex(["Trend", "Momentum", "Volatility", "Volume", "Market context"]).fillna(0)
stab_summary.to_csv(OUT / "selection_stability_summary.csv")
family_table.to_csv(OUT / "selection_by_family.csv")

fig, ax = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [3, 2]})
stability[["LASSO", "GBM"]].head(20).plot.barh(ax=ax[0], title="Share of windows in which each indicator is selected (top 20)")
ax[0].invert_yaxis()
family_table.plot.bar(ax=ax[1], rot=30, title="Average share of selected indicators by family")
plt.tight_layout(); fig.savefig(FIG / "selection_stability.png", dpi=120); plt.show()
display(stab_summary.round(2)); display(family_table.style.format("{:.0%}"))


## 6. From probabilities to portfolios
**Signal (long/flat).** About 55% of 5-day ETF returns are positive, so a signal centred on 0.5 would be long almost all the time. The signal is therefore centred on the training-window **base rate** `b` and rescaled by the model's own spread of forecasts:

$$s = \min\left(1,\ \max\left(0,\ \frac{p - b}{p_{90} - b}\right)\right)$$

where `p90` is the 90th percentile of the same window's **validation-period** forecasts. Forecasts at or below the base rate give cash; the model's most confident 10% give a full position. Both `b` and `p90` come from data before the test period.

**Floor.** If `p90 − b` is below `MIN_EDGE` (1 percentage point), the model's forecasts barely differ from the base rate, and dividing by a tiny number would turn negligible edges into full positions. In that case the strategy **stays flat for the whole test window**. Windows where this happens are listed in the edge table below.

**Sensitivity checks:** base-rate centred without rescaling, `s = max(0, (p − b)/(1 − b))`, and the original `s = max(0, 2p − 1)`.

*Why rescaling is needed:* short-horizon models produce probabilities clustered within a few points of the base rate (see `sd_p` in Section 5). Centring alone then gives positions too small to pass the 2% no-trade band. Because the rescaled signal is close to a ranking of forecasts, the **Always-on** benchmark is essential: it shows whether returns come from the model's timing or simply from being invested.

**Risk-controlled base weights (same for every strategy).** On each signal date:
1. Inverse-volatility weights from 63-day realised volatility, summing to 1.
2. Scaled by `k = min(1, 10% / σ_p)` using the 63-day covariance matrix, so the fully invested portfolio targets 10% volatility with **no leverage**.
3. Capped at 25% per ETF.

**Strategy weight** = signal × base weight. The rest is held in cash earning the T-bill rate. With every signal set to 1 this becomes the **Always-on** benchmark, which isolates the value of the forecasts from the value of volatility scaling.

**Execution.** Signals on the last trading day of each week (normally Friday), traded at the **next day's close**; changes below 2 percentage points are skipped; costs of `COST_BPS` per side are charged on turnover; weights drift with prices between rebalances.

In [ ]:
ASSETS = TICKERS + [BENCHMARK]
rets = px["Close"][ASSETS].pct_change()
rv = rets.rolling(VOL_LOOKBACK).std() * np.sqrt(252)
test_start = pred.index.get_level_values("date").min()

def period_ends(index, freq):
    s = index.to_series()
    return pd.DatetimeIndex(s.groupby(index.to_period(freq)).max().values)

weekly_dates = period_ends(dates, "W-FRI")
monthly_dates = period_ends(dates, "M")
weekly_dates = weekly_dates[(weekly_dates >= test_start) & (weekly_dates < dates[-1])]
monthly_dates = monthly_dates[(monthly_dates >= test_start) & (monthly_dates < dates[-1])]

def base_weights(t):
    # Inverse-vol weights on the tradable ETFs, scaled to the vol target (no leverage), capped per ETF.
    vol = rv.loc[t, TICKERS].dropna()
    w = (1 / vol) / (1 / vol).sum()
    cov = rets.loc[:t, w.index].iloc[-VOL_LOOKBACK:].cov() * 252
    sigma = float(np.sqrt(w @ cov @ w))
    w = w * min(1.0, VOL_TARGET / sigma)
    return w.clip(upper=MAX_WEIGHT).reindex(ASSETS).fillna(0.0)

BASE = pd.DataFrame({t: base_weights(t) for t in weekly_dates}).T

SIGNAL_COL = {"scaled": "s", "base": "sb", "raw": "sr"}   # main, base-rate only, original 2p - 1

def ml_targets(model, mode):
    s = pred[f"{SIGNAL_COL[mode]}_{model}"].unstack("ticker").reindex(index=weekly_dates, columns=ASSETS)
    s = s.fillna(0.0)                                   # missing forecast -> no position
    return (s * BASE).reindex(columns=ASSETS).fillna(0.0)

# Windows in which each model stayed flat because p90 - base rate < MIN_EDGE
print("Signal edges per window (p90 - base rate); flat = no positions that window")
display(edge_table.pivot(index="window", columns="model", values="edge").round(4))
print("Flat windows:", edge_table.loc[edge_table["flat"], ["window", "model"]].values.tolist() or "none")

In [ ]:
R = rets.fillna(0.0)
RF = rf_daily.reindex(dates).fillna(0.0)

def backtest(targets, cost_bps=COST_BPS, band=NO_TRADE_BAND, lag=EXEC_LAG):
    # Daily backtest. targets: DataFrame (signal dates x ASSETS) of target weights.
    # Each target is executed at the close `lag` trading days after its signal date.
    exec_dates = [dates[dates.get_loc(t) + lag] for t in targets.index]
    tgt = dict(zip(exec_dates, targets.to_numpy()))
    days = dates[dates > exec_dates[0]]
    r_mat, rf_vec = R.loc[days].to_numpy(), RF.loc[days].to_numpy()
    w = tgt[exec_dates[0]].copy()               # initial position set at first execution close
    turnover0 = np.abs(w).sum()
    out_r, out_to, out_exp = [], [], []
    for i, d in enumerate(days):
        r = r_mat[i]
        port = w @ r + (1 - w.sum()) * rf_vec[i]
        w = w * (1 + r) / (1 + port)            # drift with prices
        to = 0.0
        if d in tgt:
            t_w = tgt[d]
            new = np.where(np.abs(t_w - w) >= band, t_w, w)
            if new.sum() > 1:
                new = new / new.sum()
            to = np.abs(new - w).sum()
            w = new
        out_r.append(port - to * cost_bps / 1e4)
        out_to.append(to)
        out_exp.append(w.sum())
    res = pd.DataFrame({"ret": out_r, "turnover": out_to, "exposure": out_exp}, index=days)
    res.iloc[0, res.columns.get_loc("ret")] -= turnover0 * cost_bps / 1e4
    return res

In [ ]:
def const_targets(index, weights):
    return pd.DataFrame([weights.reindex(ASSETS).fillna(0.0)] * len(index), index=index)

ew = pd.Series(1 / len(TICKERS), index=TICKERS)
spy_vt = pd.DataFrame(0.0, index=weekly_dates, columns=ASSETS)
spy_vt[BENCHMARK] = (VOL_TARGET / rv.loc[weekly_dates, BENCHMARK]).clip(upper=1.0)
sma_on = (px["Close"][TICKERS].rolling(50).mean() > px["Close"][TICKERS].rolling(200).mean()).astype(float)
sma_rule = (sma_on.loc[weekly_dates] / len(TICKERS)).reindex(columns=ASSETS).fillna(0.0)

strategies = {
    "SPY buy-and-hold":       (const_targets(weekly_dates[:1], pd.Series({BENCHMARK: 1.0})), 0.0),
    "Equal-weight (monthly)": (const_targets(monthly_dates, ew), NO_TRADE_BAND),
    "Vol-targeted SPY":       (spy_vt, NO_TRADE_BAND),
    "SMA 50/200 rule":        (sma_rule, NO_TRADE_BAND),
    "Always-on (vol-scaled)": (BASE, NO_TRADE_BAND),
}
for m in MODELS:
    strategies[LABEL[m]] = (ml_targets(m, "scaled"), NO_TRADE_BAND)
    if m in ("lasso", "gbm", "lstm"):   # sensitivity signals for the main models only
        strategies[f"{LABEL[m]}, base-rate"] = (ml_targets(m, "base"), NO_TRADE_BAND)
        strategies[f"{LABEL[m]}, 2p-1"] = (ml_targets(m, "raw"), NO_TRADE_BAND)

results = {name: backtest(t, COST_BPS, band) for name, (t, band) in strategies.items()}
common = sorted(set.intersection(*(set(r.index) for r in results.values())))
results = {k: v.loc[common] for k, v in results.items()}
print(f"Backtest period: {common[0].date()} to {common[-1].date()} | {len(common)} days | cost {COST_BPS} bps/side")

## 7. Portfolio performance
Annualised return, volatility, Sharpe and Sortino ratios (excess of the T-bill rate), maximum drawdown, annual turnover (one-way, as a multiple of capital) and average invested exposure. All figures are **net of costs** and fully out-of-sample.

In [ ]:
def perf_stats(res):
    r = res["ret"]; ex = r - RF.loc[r.index]
    years = len(r) / 252
    eq = (1 + r).cumprod()
    downside = np.sqrt((np.minimum(ex, 0) ** 2).mean()) * np.sqrt(252)
    return pd.Series({
        "Ann. return": eq.iloc[-1] ** (1 / years) - 1,
        "Ann. volatility": r.std() * np.sqrt(252),
        "Sharpe": ex.mean() / ex.std() * np.sqrt(252) if ex.std() > 1e-10 else np.nan,
        "Sortino": ex.mean() * 252 / downside if downside > 1e-10 else np.nan,
        "Max drawdown": (eq / eq.cummax() - 1).min(),
        "Turnover / yr": res["turnover"].sum() / years,
        "Avg exposure": res["exposure"].mean(),
    })

perf = pd.DataFrame({k: perf_stats(v) for k, v in results.items()}).T
perf.to_csv(OUT / "portfolio_performance.csv")
perf.style.format({"Ann. return": "{:.2%}", "Ann. volatility": "{:.2%}", "Sharpe": "{:.2f}",
                   "Sortino": "{:.2f}", "Max drawdown": "{:.2%}", "Turnover / yr": "{:.1f}x",
                   "Avg exposure": "{:.0%}"})

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
show = ["SPY buy-and-hold", "Vol-targeted SPY", "Always-on (vol-scaled)"] + [LABEL[m] for m in MODELS if m in ("lasso", "gbm", "lstm")]
for k in show:
    eq = (1 + results[k]["ret"]).cumprod()
    ax[0].plot(eq, label=k); ax[1].plot(eq / eq.cummax() - 1, label=k)
ax[0].set_yscale("log"); ax[0].set_title(f"Growth of $1, net of {COST_BPS} bps per side (log scale)"); ax[0].legend()
ax[1].set_title("Drawdown"); plt.tight_layout(); fig.savefig(FIG / "equity_drawdown.png", dpi=120); plt.show()

In [ ]:
# Cost sensitivity: Sharpe ratio at 0, 2, 5 and 10 bps per side
sens_names = ["Always-on (vol-scaled)", "SMA 50/200 rule"] + [LABEL[m] for m in MODELS]
cost_sens = pd.DataFrame({c: {k: perf_stats(backtest(strategies[k][0], c, strategies[k][1]).loc[common])["Sharpe"]
                              for k in sens_names} for c in COST_GRID})
cost_sens.columns = [f"Sharpe @ {c} bps" for c in COST_GRID]

# Break-even cost: the cost per side (bps) at which the strategy's net excess return over
# T-bills falls to zero = gross annual excess return / annual one-way turnover.
def break_even(name):
    gross = backtest(strategies[name][0], 0, strategies[name][1]).loc[common]
    ex = (gross["ret"] - RF.loc[common]).mean() * 252
    turnover = gross["turnover"].sum() / (len(gross) / 252)
    return ex / turnover * 1e4 if turnover > 0.05 else np.nan

cost_sens["Break-even cost (bps)"] = [break_even(k) for k in sens_names]
cost_sens.to_csv(OUT / "cost_sensitivity.csv")
cost_sens.round(2)

## 8. Is the LSTM significantly better? Stationary block bootstrap
Null hypothesis (H4): the LSTM strategy's Sharpe ratio is **not** higher than that of LASSO or GBM, net of costs.

Daily excess returns of the two strategies are resampled **jointly** with the Politis–Romano stationary bootstrap (mean block length 20 days, 10,000 resamples), which keeps both the cross-correlation between strategies and short-run autocorrelation. The one-sided p-value is the share of resamples in which the Sharpe difference is ≤ 0. Each model is also tested against the Always-on portfolio, which asks whether its forecasts add value beyond volatility scaling.

In [ ]:
def stationary_bootstrap_indices(n, n_boot, block, rng, chunk=500):
    p = 1 / block
    for start in range(0, n_boot, chunk):
        b = min(chunk, n_boot - start)
        restart = rng.random((b, n)) < p
        restart[:, 0] = True
        starts = rng.integers(0, n, (b, n))
        pos = np.arange(n)
        last = np.maximum.accumulate(np.where(restart, pos, 0), axis=1)
        yield (np.take_along_axis(starts, last, axis=1) + (pos - last)) % n

def sharpe_rows(x):
    return x.mean(axis=1) / x.std(axis=1, ddof=1) * np.sqrt(252)

def sharpe_diff_test(a, b, n_boot=N_BOOT, block=BOOT_BLOCK, seed=SEED):
    ex_a = (results[a]["ret"] - RF.loc[common]).to_numpy()
    ex_b = (results[b]["ret"] - RF.loc[common]).to_numpy()
    if min(ex_a.std(), ex_b.std()) < 1e-10:       # a strategy that never invests has no Sharpe ratio
        return pd.Series(np.nan, index=["Sharpe A", "Sharpe B", "Difference", "95% CI low",
                                        "95% CI high", "p (A > B, one-sided)"])
    obs = sharpe_rows(ex_a[None]) - sharpe_rows(ex_b[None])
    rng = np.random.default_rng(seed)
    diffs = np.concatenate([sharpe_rows(ex_a[i]) - sharpe_rows(ex_b[i])
                            for i in stationary_bootstrap_indices(len(ex_a), n_boot, block, rng)])
    return pd.Series({"Sharpe A": sharpe_rows(ex_a[None])[0], "Sharpe B": sharpe_rows(ex_b[None])[0],
                      "Difference": obs[0], "95% CI low": np.percentile(diffs, 2.5),
                      "95% CI high": np.percentile(diffs, 97.5),
                      "p (A > B, one-sided)": (diffs <= 0).mean()})

pairs = []
for k in LSTM_KEYS:
    if k in MODELS:
        pairs += [(LABEL[k], LABEL["lasso"]), (LABEL[k], LABEL["gbm"])]
pairs += [(LABEL[m], "Always-on (vol-scaled)") for m in MODELS]
sig = pd.DataFrame({f"{a} vs {b}": sharpe_diff_test(a, b) for a, b in pairs}).T
sig.to_csv(OUT / "sharpe_bootstrap_tests.csv")
sig.round(3)

## 8b. Exposure-matched placebo (supervisor request, October 2026)
A model that is in cash most of the time can earn a good Sharpe ratio simply by being out of the market in bad periods, without any timing skill. The placebo keeps **how much** and **how long** each ETF is held, but breaks **when**.

For each draw, every ETF's actual weekly signal series is shifted circularly by its own random number of weeks k, uniform in [52, W − 52], where W is the number of decision weeks. This keeps each ETF's exposure, its distribution of position sizes and its holding lengths, so turnover and costs stay comparable, and destroys the alignment with future returns. Each draw goes through the **same backtest** as the real strategy: volatility scaling, caps, the no-trade band, costs and T-bill cash.

Reported per model: the actual Sharpe ratio; the placebo mean and 5th/95th percentiles; the placebo p-value, p = (1 + number of draws with Sharpe ≥ actual) / (N + 1); and the mean placebo exposure next to the actual exposure as a check.


In [ ]:
def signal_matrix(model):
    # The actual weekly signal per ETF (decision weeks x ETFs), as used by ml_targets.
    return pred[f"{SIGNAL_COL['scaled']}_{model}"].unstack("ticker").reindex(index=weekly_dates, columns=TICKERS).fillna(0.0)

def placebo_test(model, n_draws, rng, min_shift):
    s = signal_matrix(model)
    W = len(s)
    actual = perf.loc[LABEL[model]]
    sharpes, exposures = [], []
    for _ in range(n_draws):
        shifted = pd.DataFrame({tk: np.roll(s[tk].to_numpy(), rng.integers(min_shift, W - min_shift + 1))
                                for tk in TICKERS}, index=s.index)
        # A circular shift moves each ETF's signals in time but keeps their values.
        assert np.allclose(shifted.mean(), s.mean())
        targets = (shifted.reindex(columns=ASSETS).fillna(0.0) * BASE).reindex(columns=ASSETS).fillna(0.0)
        st = perf_stats(backtest(targets, COST_BPS, NO_TRADE_BAND).loc[common])
        sharpes.append(st["Sharpe"]); exposures.append(st["Avg exposure"])
    sharpes = np.array(sharpes)
    valid = sharpes[~np.isnan(sharpes)]
    row = {"Actual Sharpe": actual["Sharpe"],
           "Placebo mean": valid.mean() if len(valid) else np.nan,
           "Placebo 5th pct": np.percentile(valid, 5) if len(valid) else np.nan,
           "Placebo 95th pct": np.percentile(valid, 95) if len(valid) else np.nan,
           "p (placebo >= actual)": (1 + (valid >= actual["Sharpe"]).sum()) / (len(valid) + 1) if len(valid) else np.nan,
           "Actual exposure": actual["Avg exposure"],
           "Placebo mean exposure": np.mean(exposures),
           "Draws": len(valid)}
    return row, sharpes

rng = np.random.default_rng(SEED)
t0 = time.monotonic()
placebo_rows, placebo_draws = {}, {}
for m in MODELS:
    placebo_rows[LABEL[m]], placebo_draws[LABEL[m]] = placebo_test(m, N_PLACEBO, rng, PLACEBO["min_shift_weeks"])
    print(f"{LABEL[m]}: {N_PLACEBO} draws done ({time.monotonic() - t0:.0f}s)")
placebo = pd.DataFrame(placebo_rows).T
placebo.to_csv(OUT / "placebo_tests.csv")
pd.DataFrame(placebo_draws).to_csv(LARGE / "placebo_draws.csv", index=False)

ncols = min(4, len(MODELS)); nrows = -(-len(MODELS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
for ax in axes.ravel()[len(MODELS):]:
    ax.set_visible(False)
for ax, m in zip(axes.ravel(), MODELS):
    ax.hist(placebo_draws[LABEL[m]], bins=40, color="lightgrey")
    ax.axvline(placebo.loc[LABEL[m], "Actual Sharpe"], color="crimson", label="actual")
    ax.set_title(f"{LABEL[m]}\np = {placebo.loc[LABEL[m], 'p (placebo >= actual)']:.3f}", fontsize=9)
    ax.set_xlabel("Sharpe ratio, placebo draws"); ax.legend()
plt.tight_layout(); fig.savefig(FIG / "placebo_sharpe.png", dpi=120); plt.show()
placebo.round(3)


## 8c. The LSTM across training seeds
A neural network trained once can be lucky or unlucky. Each seed below is a full walk-forward of the LSTM with a different random start, put through the same signals, backtest, bootstrap and placebo. The headline is the **median and range across seeds**, not the main seed alone. Seeds come from `models.lstm.seeds` in `config.yaml`.


In [ ]:
P_COL = "p (A > B, one-sided)"
seed_rows = {}
for k, sd in LSTM_KEYS.items():
    lab = LABEL[k]
    seed_rows[sd] = {
        "AUC": pred_table.loc[lab, "AUC"],
        "Sharpe": perf.loc[lab, "Sharpe"],
        "Avg exposure": perf.loc[lab, "Avg exposure"],
        "Max drawdown": perf.loc[lab, "Max drawdown"],
        "Flat windows": int(edge_table.loc[edge_table["model"] == lab, "flat"].sum()),
        "Placebo p": placebo.loc[lab, "p (placebo >= actual)"],
        "p vs always-on": sig.loc[f"{lab} vs Always-on (vol-scaled)", P_COL],
        "p vs linear": sig.loc[f"{lab} vs {LABEL['lasso']}", P_COL],
        "p vs LightGBM": sig.loc[f"{lab} vs {LABEL['gbm']}", P_COL],
    }
lstm_seeds = pd.DataFrame(seed_rows).T.rename_axis("seed")
summary = lstm_seeds.agg(["median", "min", "max"])
summary.loc["seeds with p < 0.05"] = (lstm_seeds[["Placebo p", "p vs always-on", "p vs linear", "p vs LightGBM"]] < 0.05).sum()
lstm_seeds = pd.concat([lstm_seeds, summary])
lstm_seeds.to_csv(OUT / "lstm_seeds.csv")
lstm_seeds.round(3)


In [ ]:
# Daily net returns of every strategy (large/), and a record of what produced this run.
pd.DataFrame({k: v["ret"] for k, v in results.items()}).to_csv(LARGE / "equity_curves.csv")

run_info = {
    "label": RUN_LABEL, "commit": COMMIT, "run_date": f"{pd.Timestamp.now():%Y-%m-%d %H:%M}",
    "input_design": INPUT_DESIGN, "data_source": DATA_SOURCE, "quick_mode": QUICK_MODE,
    "data_dates": [str(dates[0].date()), str(dates[-1].date())],
    "backtest_dates": [str(common[0].date()), str(common[-1].date())],
    "test_windows": [s["label"] for s in splits],
    "seed": SEED, "lasso_C_grid": LASSO_CS, "ridge_C_grid": RIDGE_CS, "top_k_gbm": TOP_K_GBM,
    "gbm_params": GBM_PARAMS, "lstm": {"run": RUN_LSTM, "lookback": LSTM_LOOKBACK, "epochs": LSTM_EPOCHS, "seeds": LSTM_SEEDS},
    "signal": {"p_high": P_HIGH, "min_edge": MIN_EDGE},
    "portfolio": {"vol_target": VOL_TARGET, "vol_lookback": VOL_LOOKBACK, "max_weight": MAX_WEIGHT,
                  "no_trade_band": NO_TRADE_BAND, "cost_bps": COST_BPS, "cost_grid": COST_GRID},
    "bootstrap": {"n": N_BOOT, "block": BOOT_BLOCK},
    "placebo": {**PLACEBO, "draws_used": N_PLACEBO},
    "runtime_minutes": round((time.monotonic() - T_START) / 60, 1),    # excludes sleep
    "cpu_minutes": round((time.process_time() - CPU_START) / 60, 1),   # this process, all threads
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(f"Saved to {OUT}")


## 9. Initial findings and next steps

**Initial findings** *(to be written after the first full run; key numbers are saved in `results/`)*
1. **Prediction quality:** out-of-sample AUC and Brier score of each model relative to the base rate (Section 5).
2. **Probability dispersion:** how tightly probabilities cluster around the base rate, how many windows hit the `MIN_EDGE` floor, and how the rescaled signal compares with the base-rate and `2p − 1` versions (Sections 5–7).
3. **Indicator selection stability:** selection frequency per indicator, average Jaccard overlap between consecutive windows, and selection by indicator family (Section 5).
4. **Portfolio value:** net-of-cost Sharpe ratio and drawdown of each model vs SPY, vol-targeted SPY and the Always-on portfolio (Section 7).
5. **Costs:** how quickly performance decays from 0 to 10 bps, and each strategy's break-even cost.
6. **LSTM vs simpler models:** bootstrap p-values for the Sharpe differences (Section 8).

**TODO for the next stage**
- [ ] Probability calibration: reliability diagrams; compare raw vs isotonic-calibrated probabilities in position sizing.
- [ ] Ledoit–Wolf (2008) HAC test and deflated Sharpe ratio as complements to the bootstrap.
- [ ] Diebold–Mariano test on log-loss for LSTM vs LASSO/GBM forecasts.
- [ ] Hyperparameter grid for GBM and LSTM (currently fixed settings with early stopping).
- [ ] Robustness: pre/post 2018 sector reclassification; excluding the 2008 and 2020 crises; volatility targets of 8% and 12%.
- [ ] Diagnostic: correlation of LSTM signals with momentum and volatility-timing rules (complexity debate).